# rdmap walkthrough: the pretrained model on your own slides

This notebook runs `rdmap` on a folder of H&E whole-slide images with the released model
(`models/uni2-itmm-rambow-differentiation-v1`) and explores the results.

Before you start:
- run `make install`;
- put your Hugging Face token (with access to UNI2-h and CONCH) in `.env` or the `HF_TOKEN` variable;
- start Jupyter from `make shell` so the environment variables are set.

A GPU is strongly recommended for feature extraction.

In [ ]:
import os
from pathlib import Path

os.chdir(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())  # run from the repository root

WSI_DIR = "/path/to/your/slides"        # folder with .svs / .tif / ... files
METADATA = None                          # optional: "/path/to/slides.csv" with slide_id and a label column
LABEL_COLUMN = None                      # optional: name of the label column in METADATA

## 1. Run the pipeline

`rdmap run` tiles the slides, extracts features, predicts clusters, maps them onto the slides and draws
the cluster figures. Re-running the cell resumes where it stopped.

In [ ]:
from rdmap.cli import main

args = ["run", "--wsi-dir", WSI_DIR]
if METADATA:
    args += ["--metadata", METADATA, "--label-column", LABEL_COLUMN]
main(args)

## 2. Load the model and the predictions

In [ ]:
from rdmap.bundle import ModelBundle, default_model_path
from rdmap.config import Config
from rdmap.predict import CohortPredictions

cfg = Config.load()
cohort = cfg.add_adhoc_cohort(WSI_DIR, metadata=METADATA, label_column=LABEL_COLUMN)
model = ModelBundle.load(default_model_path())
out_dir = cfg.output_dir / model.name / cohort.name
predictions = CohortPredictions(cfg, cohort, model, out_dir)

print("Cluster associations:", dict(enumerate(model.associations)))
print("Results in", out_dir)

## 3. Slide-level results

`rd_high_proba` is the predicted probability of high Rambow differentiation. It is computed from the
summed presence of the high- and low-associated clusters (`presence_high`, `presence_low`).

In [ ]:
import pandas as pd

slides = pd.read_csv(out_dir / "slides.csv")
slides.sort_values("rd_high_proba", ascending=False)[
    ["slide_id", "n_tumor", "presence_high", "presence_low", "rd_high_proba", "rd_high"]
]

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(slides["presence_low"], slides["presence_high"], c=slides["rd_high_proba"], cmap="coolwarm",
           vmin=0, vmax=1)
ax.set_xlabel("presence of low-associated clusters")
ax.set_ylabel("presence of high-associated clusters")
ax.set_title("Slides coloured by rd_high_proba")

## 4. Where are the clusters on a slide?

The cluster map shows the slide thumbnail, the tiles of the associated clusters, and the slide's
cluster presences.

In [ ]:
from rdmap.plotting import select_clusters, slide_clustermap

slide = predictions.slides[0]  # pick any slide
clusters = select_clusters(model.p_values, 0.05)
slide_clustermap(predictions, slide, clusters)

## 5. Per-tile results

Each row is a tissue tile:
- `x, y, w, h`: its rectangle in level-0 pixels;
- `cluster` and `rd_association`: set for tumor tiles only.

The matching `qupath/{slide_id}.geojson` file can be dragged onto the slide in QuPath.

In [ ]:
tiles = pd.read_csv(out_dir / "tile-tables" / f"{slide.slide.slide_id}.csv")
print(tiles["rd_association"].value_counts(dropna=False))
tiles.head()

## 6. Evaluate (labelled slides only)

With a metadata CSV and label column, `evaluate` reports the AUC of `rd_high_proba`, and recomputes each
cluster's association on your cohort.

In [ ]:
if METADATA:
    main(["evaluate", "--wsi-dir", WSI_DIR, "--metadata", METADATA, "--label-column", LABEL_COLUMN])